# Aegis - Model Training & Validation
## Binary Classification for Fall Detection on Heterogeneous Edge Devices

This notebook trains our Aegis WiFi sensing model for real-world CSI data, with robust data-handling features:
1. **Mismatched subcarriers**: Standardizes both 232-subcarrier (HP) and 64-subcarrier (ESP32) inputs to a 64x500 window.
2. **Data Robustness**: Skips malformed HDF5 files and handles subjects with missing classes.
3. **Split Consistency**: Adheres to the pre-defined ID-based splits (train/val/test).
4. **Edge Ready**: Outputs a quantized TFLite model with high-confidence thresholds.

In [1]:
import numpy as np
import pandas as pd
import h5py
import os
import json
import tensorflow as tf
from tensorflow import keras
from sklearn.preprocessing import StandardScaler
from scipy import signal

# Legacy Keras for stability
os.environ['TF_USE_LEGACY_KERAS'] = '1'

# Set constants
DATA_DIR = "data/"
SUB_COUNT = 64  # Standardised for both device types
TIME_STEPS = 500
EMERGENCY_THRESHOLD = 0.85
ANOMALY_THRESHOLD = 0.60
LABEL_MAP = {"Fall": 0, "Nonfall": 1}
INV_LABEL_MAP = {0: "Fall", 1: "Nonfall"}

In [2]:
def butterworth_filter(data, lowcut=0.5, highcut=10, fs=100, order=4):
    """Apply Butterworth bandpass filter to isolate human movement"""
    nyq = 0.5 * fs
    low = lowcut / nyq
    high = highcut / nyq
    b, a = signal.butter(order, [low, high], btype="band")
    return signal.filtfilt(b, a, data, axis=1)

def load_csi_sample(file_path):
    """Robust HDF5 loading with shape standardization"""
    full_path = os.path.join(DATA_DIR, file_path.lstrip("./"))
    try:
        with h5py.File(full_path, "r") as f:
            data = f["CSI_amps"][:]
            data = data.squeeze() 
            if data.shape[0] >= SUB_COUNT:
                data = data[:SUB_COUNT, :]
            else:
                return None
            if data.shape[1] < TIME_STEPS:
                return None
            data = data[:, :TIME_STEPS]
            data = butterworth_filter(data)
            return data.T 
    except Exception:
        return None

def load_dataset_from_ids(ids_file, metadata_df):
    """Loads all valid samples belonging to a split ID set"""
    with open(os.path.join(DATA_DIR, "splits", ids_file), "r") as f:
        ids = json.load(f)
    X, y = [], []
    for sample_id in ids:
        row = metadata_df[metadata_df["id"] == sample_id]
        if row.empty: continue
        sample = load_csi_sample(row.iloc[0]["file_path"])
        if sample is not None:
            X.append(sample)
            y.append(LABEL_MAP[row.iloc[0]["label"]])
    return np.array(X), np.array(y)

metadata_df = pd.read_csv(os.path.join(DATA_DIR, "metadata/sample_metadata.csv"))
X_train, y_train = load_dataset_from_ids("train_id.json", metadata_df)
X_val, y_val = load_dataset_from_ids("val_id.json", metadata_df)
X_test, y_test = load_dataset_from_ids("test_id.json", metadata_df)
print(f"Loaded Train: {len(X_train)}, Val: {len(X_val)}, Test: {len(X_test)}")

Loaded Train: 4619, Val: 994, Test: 985


In [3]:
X_train_flat = X_train.reshape(X_train.shape[0], -1)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train_flat).reshape(X_train.shape)
X_val_scaled = scaler.transform(X_val.reshape(X_val.shape[0], -1)).reshape(X_val.shape)
X_test_scaled = scaler.transform(X_test.reshape(X_test.shape[0], -1)).reshape(X_test.shape)
y_train_cat = keras.utils.to_categorical(y_train, 2)
y_val_cat = keras.utils.to_categorical(y_val, 2)
y_test_cat = keras.utils.to_categorical(y_test, 2)

# Export scaler parameters for Edge Node
scaler_params = {
    "mean": scaler.mean_.tolist(),
    "scale": scaler.scale_.tolist()
}
with open('scaler_params.json', 'w') as f:
    json.dump(scaler_params, f)
print("✅ Scaler parameters exported to scaler_params.json")

✅ Scaler parameters exported to scaler_params.json


In [4]:
model = keras.Sequential([
    keras.layers.Conv1D(32, 5, activation="relu", input_shape=(TIME_STEPS, SUB_COUNT)),
    keras.layers.MaxPooling1D(2),
    keras.layers.Conv1D(64, 3, activation="relu"),
    keras.layers.GlobalAveragePooling1D(),
    keras.layers.Dense(64, activation="relu"),
    keras.layers.Dropout(0.4),
    keras.layers.Dense(2, activation="softmax")
])
model.compile(optimizer="adam", loss="categorical_crossentropy", metrics=["accuracy"])
model.summary()

Model: "sequential"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 conv1d (Conv1D)             (None, 496, 32)           10272     
                                                                 
 max_pooling1d (MaxPooling1  (None, 248, 32)           0         
 D)                                                              
                                                                 
 conv1d_1 (Conv1D)           (None, 246, 64)           6208      
                                                                 
 global_average_pooling1d (  (None, 64)                0         
 GlobalAveragePooling1D)                                         
                                                                 
 dense (Dense)               (None, 64)                4160      
                                                                 
 dropout (Dropout)           (None, 64)                0

2026-03-03 05:02:11.083131: I metal_plugin/src/device/metal_device.cc:1154] Metal device set to: Apple M4
2026-03-03 05:02:11.083165: I metal_plugin/src/device/metal_device.cc:296] systemMemory: 16.00 GB
2026-03-03 05:02:11.083182: I metal_plugin/src/device/metal_device.cc:313] maxCacheSize: 5.33 GB
2026-03-03 05:02:11.083209: I tensorflow/core/common_runtime/pluggable_device/pluggable_device_factory.cc:306] Could not identify NUMA node of platform GPU ID 0, defaulting to 0. Your kernel may not have been built with NUMA support.
2026-03-03 05:02:11.083225: I tensorflow/core/common_runtime/pluggable_device/pluggable_device_factory.cc:272] Created TensorFlow device (/job:localhost/replica:0/task:0/device:GPU:0 with 0 MB memory) -> physical PluggableDevice (device: 0, name: METAL, pci bus id: <undefined>)


In [5]:
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint

callbacks = [
    EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True, verbose=1),
    ModelCheckpoint('best_aegis_model.h5', monitor='val_loss', save_best_only=True, verbose=1)
]

history = model.fit(
    X_train_scaled, y_train_cat, 
    validation_data=(X_val_scaled, y_val_cat), 
    epochs=50, 
    batch_size=32,
    callbacks=callbacks
)

Epoch 1/50
  1/145 [..............................] - ETA: 11:47 - loss: 0.8070 - accuracy: 0.6250

2026-03-03 05:02:17.226524: I tensorflow/core/grappler/optimizers/custom_graph_optimizer_registry.cc:117] Plugin optimizer for device_type GPU is enabled.


145/145 [==============================] - ETA: 0s - loss: 0.6609 - accuracy: 0.6590
Epoch 1: val_loss improved from inf to 0.58050, saving model to best_aegis_model.h5
145/145 [==============================] - 7s 16ms/step - loss: 0.6609 - accuracy: 0.6590 - val_loss: 0.5805 - val_accuracy: 0.6700
Epoch 2/50
 11/145 [=>............................] - ETA: 1s - loss: 0.6036 - accuracy: 0.7244

/opt/anaconda3/envs/aegis-m4/lib/python3.10/site-packages/tf_keras/src/engine/training.py:3098: UserWarning: You are saving your model as an HDF5 file via `model.save()`. This file format is considered legacy. We recommend using instead the native TF-Keras format, e.g. `model.save('my_model.keras')`.
  saving_api.save_model(


141/145 [============================>.] - ETA: 0s - loss: 0.5793 - accuracy: 0.7411
Epoch 2: val_loss improved from 0.58050 to 0.50303, saving model to best_aegis_model.h5
145/145 [==============================] - 2s 11ms/step - loss: 0.5789 - accuracy: 0.7415 - val_loss: 0.5030 - val_accuracy: 0.7374
Epoch 3/50
142/145 [============================>.] - ETA: 0s - loss: 0.5557 - accuracy: 0.7623
Epoch 3: val_loss improved from 0.50303 to 0.49059, saving model to best_aegis_model.h5
145/145 [==============================] - 2s 10ms/step - loss: 0.5539 - accuracy: 0.7627 - val_loss: 0.4906 - val_accuracy: 0.7757
Epoch 4/50
145/145 [==============================] - ETA: 0s - loss: 0.5077 - accuracy: 0.7850
Epoch 4: val_loss did not improve from 0.49059
145/145 [==============================] - 1s 10ms/step - loss: 0.5077 - accuracy: 0.7850 - val_loss: 0.5046 - val_accuracy: 0.7696
Epoch 5/50
141/145 [============================>.] - ETA: 0s - loss: 0.5113 - accuracy: 0.7877
Epoch 5:

In [6]:
model.load_weights('best_aegis_model.h5')

converter = tf.lite.TFLiteConverter.from_keras_model(model)
converter.optimizations = [tf.lite.Optimize.DEFAULT]
tflite_model = converter.convert()

with open('aegis_wave_final.tflite', 'wb') as f:
    f.write(tflite_model)

print(f"✅ TFLite model size: {len(tflite_model) / 1024:.2f} KB")
print(f"Suitable for: Raspberry Pi, ESP32, edge routers")

INFO:tensorflow:Assets written to: /var/folders/xk/q3wl5lb50lvgqn4fmglj9z6r0000gn/T/tmpxd5sawo9/assets


INFO:tensorflow:Assets written to: /var/folders/xk/q3wl5lb50lvgqn4fmglj9z6r0000gn/T/tmpxd5sawo9/assets


✅ TFLite model size: 27.34 KB
Suitable for: Raspberry Pi, ESP32, edge routers


2026-03-03 05:02:49.124856: W tensorflow/compiler/mlir/lite/python/tf_tfl_flatbuffer_helpers.cc:378] Ignored output_format.
2026-03-03 05:02:49.124865: W tensorflow/compiler/mlir/lite/python/tf_tfl_flatbuffer_helpers.cc:381] Ignored drop_control_dependency.
2026-03-03 05:02:49.125061: I tensorflow/cc/saved_model/reader.cc:83] Reading SavedModel from: /var/folders/xk/q3wl5lb50lvgqn4fmglj9z6r0000gn/T/tmpxd5sawo9
2026-03-03 05:02:49.125644: I tensorflow/cc/saved_model/reader.cc:51] Reading meta graph with tags { serve }
2026-03-03 05:02:49.125648: I tensorflow/cc/saved_model/reader.cc:146] Reading SavedModel debug info (if present) from: /var/folders/xk/q3wl5lb50lvgqn4fmglj9z6r0000gn/T/tmpxd5sawo9
2026-03-03 05:02:49.126741: I tensorflow/compiler/mlir/mlir_graph_optimization_pass.cc:388] MLIR V1 optimization pass is not enabled
2026-03-03 05:02:49.127362: I tensorflow/cc/saved_model/loader.cc:233] Restoring SavedModel bundle.
2026-03-03 05:02:49.150280: I tensorflow/cc/saved_model/loader.